# 1D J1J2=0.8: PoincareRNN

This is part of the work https://arxiv.org/abs/2604.24337.

- Use norm_clamp = 0.618 WITH added norm_clamp inside one_rnn_transform at 2 places).
- When norm_clamp = 0.78, or 0.618 with no additional norm_clamps inside one_rnn_transform, numerical explosion happened where the states are all pushed to the boundary, with norm = 0.998 or larger. 

In [1]:
import os
import sys
import time
sys.path.append('../utility')
from j1j2_hyprnn_train_loop import *
#from j1j2_tau_hyprnn_train_loop import *

GPU Available:  False


In [2]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

In [3]:
E_exact = -42.07006297371643
units = 70
syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.8
nsteps = 1001
var_tol = 20.0

# With tau (sample generation)

In [4]:
rmax=0.7
fname=f'1d_J1J2_results_N={syssize}_tau_hypRNN/rmax={rmax}/J2={J2}'

hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
for name, param in hRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
    
t0=time.time()
mhE, vhE = run_J1J2(hRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=2e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([2, 70])
Layer: rnn.b | Size: torch.Size([1, 70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])


/Users/hl/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/Users/hl/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -0.94458, mean energy: 44.33379-0.00013j, varE: 0.41392| Max Radius: 0.0144 | Hyp LR: 2.00e-03| LR: 5.00e-03|tau=1.05
Best model saved at epoch 1 with best E=44.28249-0.00661j, varE=0.62962
Best model saved at epoch 2 with best E=43.73523-0.03738j, varE=2.49193
Best model saved at epoch 3 with best E=41.76496-0.23655j, varE=7.67230
step: 10, loss: 3.75769, mean energy: 4.86321+0.19393j, varE: 44.56815| Max Radius: 0.5118 | Hyp LR: 2.00e-03| LR: 5.00e-03|tau=1.0490000000000002
step: 20, loss: 9.79507, mean energy: -1.99508+0.53944j, varE: 40.78624| Max Radius: 0.7524 | Hyp LR: 2.00e-03| LR: 5.00e-03|tau=1.048
step: 30, loss: -21.97687, mean energy: -3.39004-0.40353j, varE: 49.12483| Max Radius: 0.8837 | Hyp LR: 2.00e-03| LR: 5.00e-03|tau=1.0470000000000002
step: 40, loss: 2.07532, mean energy: -9.10385-0.47729j, varE: 27.26751| Max Radius: 0.9118 | Hyp LR: 2.00e-03| LR: 5.00e-03|tau=1.046
step: 50, loss: -6.28589, mean energy: -11.23539-0.28661j, varE: 27.68316| Max Radiu

Best model saved at epoch 329 with best E=-31.53044+0.06859j, varE=10.46827
step: 330, loss: -2.82410, mean energy: -30.97899-0.12120j, varE: 11.15336| Max Radius: 0.9392 | Hyp LR: 5.00e-04| LR: 1.25e-03|tau=1.0170000000000001
step: 340, loss: -15.39490, mean energy: -30.71302-0.02048j, varE: 11.88405| Max Radius: 0.9380 | Hyp LR: 5.00e-04| LR: 1.25e-03|tau=1.016
step: 350, loss: 1.71118, mean energy: -30.50175-0.16567j, varE: 15.48842| Max Radius: 0.9378 | Hyp LR: 5.00e-04| LR: 1.25e-03|tau=1.0150000000000001
Best model saved at epoch 360 with best E=-31.53884+0.02778j, varE=10.99533
step: 360, loss: 0.59631, mean energy: -31.53884+0.02778j, varE: 10.99533| Max Radius: 0.9390 | Hyp LR: 5.00e-04| LR: 1.25e-03|tau=1.014
Best model saved at epoch 361 with best E=-31.71981+0.36418j, varE=13.68158
Best model saved at epoch 367 with best E=-32.03059-0.25142j, varE=13.46415
step: 370, loss: 13.89014, mean energy: -31.31806-0.19653j, varE: 16.25467| Max Radius: 0.9388 | Hyp LR: 5.00e-04| LR: 

step: 750, loss: -1.15210, mean energy: -34.54788+0.04186j, varE: 13.29301| Max Radius: 0.9387 | Hyp LR: 6.25e-05| LR: 1.56e-04|tau=1.0
step: 760, loss: -0.45203, mean energy: -35.05000-0.13270j, varE: 15.72866| Max Radius: 0.9386 | Hyp LR: 3.13e-05| LR: 7.81e-05|tau=1.0
step: 770, loss: -4.34119, mean energy: -35.46230+0.01026j, varE: 12.92228| Max Radius: 0.9380 | Hyp LR: 3.13e-05| LR: 7.81e-05|tau=1.0
step: 780, loss: -17.26135, mean energy: -35.68174-0.58810j, varE: 58.08890| Max Radius: 0.9373 | Hyp LR: 3.13e-05| LR: 7.81e-05|tau=1.0
step: 790, loss: -7.59082, mean energy: -35.54242-0.10474j, varE: 9.63215| Max Radius: 0.9379 | Hyp LR: 3.13e-05| LR: 7.81e-05|tau=1.0
step: 800, loss: -2.06946, mean energy: -34.13987+0.03629j, varE: 19.27560| Max Radius: 0.9386 | Hyp LR: 3.13e-05| LR: 7.81e-05|tau=1.0
step: 810, loss: 4.49451, mean energy: -35.98917+0.07314j, varE: 10.55422| Max Radius: 0.9358 | Hyp LR: 1.56e-05| LR: 3.91e-05|tau=1.0
Best model saved at epoch 818 with best E=-36.074

# No tau (in sample generation)

In [4]:
rmax=0.618
fname=f'1d_J1J2_results_N={syssize}_hypRNN/rmax={rmax}/J2={J2}'
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
for name, param in hRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")

t0=time.time()
mhE, vhE = run_J1J2(hRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([2, 70])
Layer: rnn.b | Size: torch.Size([1, 70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -0.94458, mean energy: 44.33379-0.00013j, varE: 0.41392| Max Radius: 0.0144 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=44.28251-0.00661j, varE=0.62957
Best model saved at epoch 2 with best E=43.73513-0.03737j, varE=2.49202
Best model saved at epoch 3 with best E=41.76025-0.20893j, varE=7.80622
step: 10, loss: -2.58234, mean energy: 7.92917+0.16749j, varE: 50.51993| Max Radius: 0.6409 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 20, loss: 7.13560, mean energy: -2.09656-0.52370j, varE: 33.04150| Max Radius: 0.8342 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 30, loss: -2.28979, mean energy: -5.28831-0.10025j, varE: 38.45785| Max Radius: 0.8648 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 40, loss: 1.89948, mean energy: -10.27500-0.11667j, varE: 29.26104| Max Radius: 0.8745 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 50, loss: -0.98798, mean energy: -10.53180-0.09385j, varE: 29.33691| Max Radius: 0.8848 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 60, loss: -1.67438, mean 

### rmax=0.99

In [4]:
rmax=0.99
fname=f'1d_J1J2_results_N={syssize}_hypRNN/rmax={rmax}/J2={J2}'
hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
for name, param in hRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")

t0=time.time()
mhE, vhE = run_J1J2(hRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([2, 70])
Layer: rnn.b | Size: torch.Size([1, 70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -0.94458, mean energy: 44.33379-0.00013j, varE: 0.41392| Max Radius: 0.0144 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=44.28251-0.00661j, varE=0.62957
Best model saved at epoch 2 with best E=43.73513-0.03737j, varE=2.49202
Best model saved at epoch 3 with best E=41.76025-0.20893j, varE=7.80622
step: 10, loss: 152.96808, mean energy: 15.56368+0.19297j, varE: 30.98349| Max Radius: 0.9945 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 20, loss: -10.35329, mean energy: -5.23160-0.18687j, varE: 26.77470| Max Radius: 0.9963 | Hyp LR: 1.00e-03| LR: 5.00e-03
Best model saved at epoch 23 with best E=-4.67076-0.26737j, varE=18.84502
Best model saved at epoch 27 with best E=-5.32202+0.20461j, varE=19.08840
step: 30, loss: -10.51943, mean energy: -4.82210+0.30969j, varE: 12.67242| Max Radius: 0.9968 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 40, loss: -1.09435, mean energy: -4.89356+0.10125j, varE: 0.51114| Max Radius: 0.9964 | Hyp LR: 1.00e-03| LR: 5.00e-03
step: 

### rmax=0.7, lr2=2e-3: -36 (best)

In [7]:
rmax=0.7
fname=f'1d_J1J2_results_N={syssize}_hypRNN/rmax={rmax}/J2={J2}'

hRNN = RNNwavefunction_hyp(systemsize=syssize, cell_type='HypRNN', bias_geom='hyp',
                           hyp_non_lin='id', units=units,r_max=rmax, seed=111)
for name, param in hRNN.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
    
t0=time.time()
mhE, vhE = run_J1J2(hRNN, nsteps, syssize, var_tol, J1_=J1, J2_=J2, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=2e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([70, 70])
Layer: rnn.U | Size: torch.Size([2, 70])
Layer: rnn.b | Size: torch.Size([1, 70])
Layer: dense_a.weight | Size: torch.Size([2, 70])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 70])
Layer: dense_p.bias | Size: torch.Size([2])


/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/opt/anaconda3/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -0.94458, mean energy: 44.33379-0.00013j, varE: 0.41392| Max Radius: 0.0144 | Hyp LR: 2.00e-03| LR: 5.00e-03
Best model saved at epoch 1 with best E=44.28249-0.00661j, varE=0.62962
Best model saved at epoch 2 with best E=43.73523-0.03738j, varE=2.49193
Best model saved at epoch 3 with best E=41.76496-0.23655j, varE=7.67230
step: 10, loss: 3.75769, mean energy: 4.86321+0.19393j, varE: 44.56815| Max Radius: 0.5118 | Hyp LR: 2.00e-03| LR: 5.00e-03
step: 20, loss: 9.79507, mean energy: -1.99508+0.53944j, varE: 40.78624| Max Radius: 0.7524 | Hyp LR: 2.00e-03| LR: 5.00e-03
step: 30, loss: -21.97687, mean energy: -3.39004-0.40353j, varE: 49.12483| Max Radius: 0.8837 | Hyp LR: 2.00e-03| LR: 5.00e-03
step: 40, loss: 2.07532, mean energy: -9.10385-0.47729j, varE: 27.26751| Max Radius: 0.9118 | Hyp LR: 2.00e-03| LR: 5.00e-03
step: 50, loss: -6.28589, mean energy: -11.23539-0.28661j, varE: 27.68316| Max Radius: 0.9266 | Hyp LR: 2.00e-03| LR: 5.00e-03
step: 60, loss: -3.57050, mean e